<a href="https://colab.research.google.com/github/mmferes/PPD-2026/blob/main/Aula4/praticaguiada_0704.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Gerenciamento de *threads* com *pthreads*
Créditos: Prof. Dr. Hélio


1.   Item da lista
2.   Item da lista




# pthreads

A biblioteca padrão para o gerenciamento de *threads* em sistemas compatíveis com POSIX, como Linux e BSDs, por exemplo, é [pthreads](https://pubs.opengroup.org/onlinepubs/007908775/xsh/pthread.h.html).


## Criação de novas threads

A criação de *threads* é feita com a chamada [pthread_create](https://man7.org/linux/man-pages/man3/pthread_create.3.html)():
```
pthread_create(pthread_t *restrict thread, const pthread_attr_t *restrict attr,
               void *(*start_routine)(void *), void *restrict arg);
```

Alguns aspectos a observar nesta chamada:

* o código de uma *thread* é definido por uma **função** dentro do código do processo.
* A função da *thread* tem um único parâmetro, que é um ponteiro para *void*
* O valor de retorno da *thread* é também um ponteiro para *void*
* Ao concluir a execução do código da função especificada, a tarefa criada (thread) termina, sem retornar ao ponto do código que chamou pthread_create()

Embora ter os parâmetros e o valor de retorno definidos desta forma (void *) possa parecer algo restritivo, esse modelo oferece grande flexibilidade, já que, em C, pode-se converter esses parâmetros para ponteiros para qualquer tipo de variável, inclusive para estruturas.

O **identificador da nova *thread*** criada é salvo no endereço de memória passado no primeiro parâmetro da chamada.

A criação de *threads* pode ocorrer em qualquer ponto em um programa. Nos casos mais simples, um programa que manipula *threads* vai fazer chamadas à função *pthread_create* para criá-las, a partir da função *main*.

Ah, vale observar o valor de retorno da chamada de criação: **0 = sucesso**, outro valor = código do erro ocorrido.


## Identificador de uma *thread*

Assim como o valor **pid** identifica unicamente um processo para o SO, cada *thread* tem também um identificador único, chamado ***thread ID***. Como mencionado, esse valor é retornado como um dos parâmetros da chamada *pthread_create*. Além disso, uma *thread* pode obter o seu idenficador usando a chamada ***pthread_self***(), que é equivalente à chamada ***getpid***() para processos.

Observe que todas as *threads* de um processo compartilham o mesmo identificador de processo. Assim, a execução da chamada *getpid*() por qualquer *thread* de um processo deve retornar o mesmo valor.

## Encerramento da execução de *threads*

Depois de criar novas *threads* no programa, a função *main* não pode simplesmente terminar, com *return* ou *exit*(), pois isso faria com que o processo todo, com todas as suas *threads*, fosse **terminado**.

Assim, é possível que a função *main*, que é a *thread* original associada a este processo, passe à execução de algum códido com duração equivalente à das demais *threads*. Também é comum que a função *main* seja usada apenas como uma coordenadora das atividades do programa. Neste caso, ela pode simplesmente esperar pela conclusão das demais *threads*.

Ao concluir suas atividades, uma função que corresponde ao código de uma *thread* pode realizar o **retorno de algum valor**, ou pode usar uma chamada explícita à função ***pthread_exit***(). Em ambos os casos, é possível retornar um ponteiro para alguma posição de memória.

Para efeitos de sincronização e/ou para saber o resultado da execução de uma outra *thread* do mesmo processo, é possível usar a chamada [***pthread_join***](https://man7.org/linux/man-pages/man3/pthread_join.3.html)(). De maneira parecida com a chamada *waitpid*(), *pthread_join* serve para esperar pelo retorno (conclusão) de uma *thread* específica.

Qualquer *thread* de um processo pode usar *pthread_join* para esperar por qualquer outra *thread* deste processo; isso não precisa ser feito pela função *main* ou pela *thread* que criou a *thread* que se quer esperar. Na chamada, contudo, é preciso fornecer o identificador da *thread* esperada.



O programa exemplo a seguir ilustra a criação de *threads* com *pthreads* (*POSIX threads*).

## Compilando programas com *pthreads* com **gcc**

Só um comentário rápido aqui. Para compilar com **gcc** programas C que usem as funções da API ***pthreads***, é preciso incluir o parâmetro ***-pthread*** na linha de comando.

Ex:
```
$ gcc prog.c -o prog -pthread
```

Para saber mais sobre isso, vejam o [manual](https://man7.org/linux/man-pages/man1/gcc.1.html):

       -pthread
           Define additional macros required for using the POSIX threads
           library.  You should use this option consistently for both
           compilation and linking.

In [ ]:
%%writefile t1.c

/*
 * Objetivo:
   Observar a criação de threads e o término de suas execuções.
   pthread_exit() ou return indicam fim de thread.
   Main thread deve esperar threads terminarem, ou sair com pthread_exit().
   Se função main terminar (return, exit ou _exit), processo todo (todas as suas
   threads) termina.
 */

#include <pthread.h>
#include <stdio.h>
#include <string.h>
#include <unistd.h>

#define NUM_THREADS	4

void *
hello_w(void *arg)
{
  pthread_t tid = pthread_self();

  sleep(5);

	printf("Um alo^ da thread %lu, parte do processo %d\n",
        (long unsigned)tid, (int)getpid() );

	pthread_exit(NULL);
}

int main (int argc, char *argv[])
{
	int t, status;

  // vetor de pthread_t para salvamento dos identificadores das threads
	pthread_t threads[NUM_THREADS];

  printf("Processo %d vai criar threads...\n\n", getpid() );

	for (t=0; t < NUM_THREADS; t++) {

		status = pthread_create(&threads[t], NULL, hello_w, NULL);

		if (!status) {
      printf("main criou thread %d (%lu)\n",t,(long unsigned)threads[t]);
    } else {
			printf("Falha da criacao da thread %d (%d)\n",t,status);
			return (1);
		}
	}

// pthread_exit(NULL);
 //return 0;
 //sleep(1);

  // Se função main, ou qualquer thread, terminar com exit, processo todo termina
  // Em geral, main executa algo e, ao final, espera demais threads acabarem antes
  // de terminar o programa. Espera pela conclusão é feita com pthread_join()

  // loop de espera pelo término da execução das threads
  for (t=0; t < NUM_THREADS; t++) {

    // join ignorando o valor de retorno
    status = pthread_join(threads[t], NULL);

    if (! status) {
      printf("Thread %d (%lu) retornou\n",t,(long unsigned)threads[t]);
    } else{
      fprintf(stderr,"Erro em pthread_join (%d)\n",status);
      return (2);
    }
  }

  return 0;
}

Overwriting t1.c


In [ ]:
! if [ ! t1 -nt t1.c ]; then gcc t1.c -o t1 -pthread; fi
! ./t1

Processo 10303 vai criar threads...

main criou thread 0 (132143109678656)
main criou thread 1 (132143101285952)
main criou thread 2 (132143092893248)
main criou thread 3 (132143084500544)
Um alo^ da thread 132143109678656, parte do processo 10303
Um alo^ da thread 132143092893248, parte do processo 10303
Um alo^ da thread 132143084500544, parte do processo 10303
Um alo^ da thread 132143101285952, parte do processo 10303
Thread 0 (132143109678656) retornou
Thread 1 (132143101285952) retornou
Thread 2 (132143092893248) retornou
Thread 3 (132143084500544) retornou


**Questões**:

* Viram que todas as *threads* exibem o mesmo ***pid***?
* Viram os valores dos ***thread_ids***?
* Por que as impressões das mensagens das *threads* podem aparecer fora de ordem, em ordem diferente a cada execução?


Pensando nos *loops* de criação e de espera pelo término de *threads*, o programa a seguir, que junta esses dois *loops*, tem sentido? Por quê?

In [ ]:
%%writefile t2.c

#include <pthread.h>
#include <stdio.h>
#include <unistd.h>

#define NUM_THREADS	4

void *
hello_w(void *arg)
{
  printf("Thread %ld trabalhando...\n", (long int)pthread_self() );
  sleep(1);
	pthread_exit(NULL);
}

int main (int argc, char *argv[])
{
	int t;

  // vetor de pthread_t para salvamento dos identificadores das threads
	pthread_t threads[NUM_THREADS];

	for (t=0; t < NUM_THREADS; t++) {

 		if (pthread_create(&threads[t], NULL, hello_w, NULL)) {
      printf("Falha da criacao da thread %d\n",t);
			return(1);
		}

    if (pthread_join(threads[t], NULL) ) {
      printf("Erro em pthread_join, esperando pela thread %d\n",t);
      return(2);
    }
  }

  return 0;
}

Writing t2.c


In [ ]:
! if [ ! t2 -nt t2.c ]; then gcc -Wall t2.c -o t2 -pthread; fi
! ./t2

Thread 134022229083712 trabalhando...
Thread 134022229083712 trabalhando...
Thread 134022229083712 trabalhando...
Thread 134022229083712 trabalhando...


# Passando parâmetros na ativação e no retorno das *threads*

Vejamos como funciona a passagem de parâmetros para a função da *thread*, tanto em sua criação quanto no retorno da função associada.

Como visto, a chamada *pthread_create* prevê apenas um parâmetro na função da *thread*. Sendo este um ponteiro sem tipo definido ( void *), em C, isso pode ser usado para passar ponteiros para qualquer posição de memória, que contenha qualquer tipo de dado!

Ah, considerando as arquiteturas 64 bits atuais, em que um endereço de memória tem 64 bits, também é possível passar como parâmetro para a função da *thread* qualquer valor que caiba em (<=) 8 bytes.

<br>

Uma estratégia para acomodar funções que precisam múltiplos parâmetros é definir um tipo de estrutura (***struct***) contendo esses parâmetros. Daí, para a chamada de criação da *thread*, preenche-se uma estrutura deste tipo com os parâmetros desejados e passa-se o endereço desta estrutura como parâmetro da chamada. Ao receber este parâmetro, basta que a função da *thread*  ajuste a referência para cada campo da estrutura recebida.

<br>

Nos casos de uso de *threads* para programação paralela com decomposição de dados, comumente cria-se várias *threads* da mesma função, com o objetivo que cada uma delas manipule uma parte distinta dos dados. Para isso, uma estratégia comum é passar para a função da *thread* apenas um identificador lógico desta *thread*. Esse identificador pode então ser usado para que a própria *thread* determine qual parte dos dados irá manipular.

<br>

Quanto ao valor de retorno, as mesmas considerações são válidas sobre o uso de um ponteiro. No código de chamada da função *pthread_join*, é preciso estar atento que, originialmente, deverá ser fornecido um endereço de uma posição de memória **onde será salvo um endereço** (void **).

Por fim, como a linguagem C é bastante flexível, basta que o programa passe parâmetros que são tratados de maneira apropriada nas funções que emitem e processam as chamadas.

In [ ]:
%%writefile t3.c

#include <pthread.h>
#include <stdio.h>
#include <string.h>
#include <unistd.h>

#define NUM_THREADS	4

// declare aqui, como variávies globais, as variáveis que serão
// compartilhadas entre as threads deste processo

void *
hello_w(void *arg)
{
  int * num = (int *)arg;
  int ind = *num;

	printf("Thread %d trabalhando...\n", ind);
 //	printf("Thread %d trabalhando...\n", *(int *)arg);

  sleep(1);

	pthread_exit(NULL);
}

int main (int argc, char *argv[])
{
	int t, status;

  // vetor de pthread_t para salvamento dos identificadores das threads
	pthread_t threads[NUM_THREADS];

  // vetor de inteiros para uso como parâmetros para as threads
  int args[NUM_THREADS];

	for (t=0; t < NUM_THREADS; t++) {

    args[t]=t;  // ajuste do parâmetro para a thread; neste caso, um índice lógico

		status = pthread_create(&threads[t], NULL, hello_w, (void *)&args[t]);

		if (!status) {
      printf("main criou thread %d\n",t);
    } else {
			printf("Falha da criacao da thread %d (%d)\n",t,status);
			return (1);
		}
	}

  // loop de espera pelo término da execução das threads
  for (t=0; t < NUM_THREADS; t++) {

    // join ignorando o valor de retorno
    status = pthread_join(threads[t], NULL);

    if (! status) {
      printf("Thread %d (%lu) retornou\n",t,(long unsigned)threads[t]);
    } else{
      printf("Erro em pthread_join da thread %d (%d)\n",t,status);
      return (2);
    }
  }

  return 0;
}

Overwriting t3.c


In [ ]:
! if [ ! t3 -nt t3.c ]; then gcc -Wall t3.c -o t3 -pthread; fi
! ./t3

main criou thread 0
Thread 0 trabalhando...
Thread 0 trabalhando...
main criou thread 1
main criou thread 2
Thread 1 trabalhando...
Thread 1 trabalhando...
Thread 2 trabalhando...
Thread 2 trabalhando...
main criou thread 3
Thread 3 trabalhando...
Thread 3 trabalhando...
Thread 0 (133616138167872) retornou
Thread 1 (133616129775168) retornou
Thread 2 (133616121382464) retornou
Thread 3 (133616112989760) retornou


Criando diferentes tipos de *trheads*, com diferentes parâmetros...

In [ ]:
%%writefile t4.c
/*
** Objetivo:
    observar que o tipo *void usado no argumento da função da thread pode
    ser qualquer coisa. É preciso apenas que o argumento seja usado considerando
    o tipo que foi passado.
    Não é qualquer tipo de parâmetro que pode ser tratado como um ponteiro, contudo.
    No caso da estrutura, é preciso passar o seu endereço.
*/

#include <pthread.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>

#define LEN 64

// exemplo de estrutura de dados para usar como parâmetos na criação
// de thread usando função que precisa de vários parâmetros
typedef struct par {
	int num_int;
	char num_char[LEN];
} st_par;


// exemplo de função que espera um valor inteiro (long int) como parâmetro
void * f_int(void *num_int)
{
	printf("Aloˆ, aqui é %ld\n",(long int)num_int);

	// pthread_exit(NULL);
	pthread_exit((long int *)1);
}

// exemplo de função que espera uma string (vetor de char) como parâmetro
void * f_char(void *num_char)
{
	printf("Aqui é thread %s\n",(char *)num_char);

	// pthread_exit(NULL);
	pthread_exit((long int *)2);
}

// exemplo de função que precisa de vários parâmetros
void *
f_struct(void *num_struct)
{
	st_par *param;
	param=(st_par*)num_struct;
	// int *intp;
	// char *charp;

	// intp = &(param->num_int);
	// charp = &(param->num_char);

	printf("Ola' da thread %d %s\n",param->num_int,param->num_char);

	// pthread_exit(NULL);
	pthread_exit((long int *)34);
}

int
main (int argc, char *argv[])
{
	pthread_t th_int, th_char, th_struct;
	int status, *retval;
	long int num_int=1;
	char *num_char="dois";
	st_par num_struct;
	char err_msg[LEN];

	num_struct.num_int=3;
	strcpy(num_struct.num_char,"quatro");

	if((status=pthread_create(&th_int, NULL, f_int, (void *)num_int))) {
		strerror_r(status,err_msg,LEN);
		printf("Erro criando th_int: %s\n",err_msg);
		exit(EXIT_FAILURE);
	}
	if((status=pthread_create(&th_char, NULL, f_char, (void *)num_char))) {
		strerror_r(status,err_msg,LEN);
		printf("Erro criando th_char: %s\n",err_msg);
		exit(EXIT_FAILURE);
	}
	if((status=pthread_create(&th_struct, NULL, f_struct, (void *)&num_struct))) {
		strerror_r(status,err_msg,LEN);
		printf("Erro criando th_struct: %s\n",err_msg);
		exit(EXIT_FAILURE);
	}

	if((status = pthread_join(th_int, (void **)&retval))) {
		strerror_r(status,err_msg,LEN);
		printf("Erro em pthread_join th_int: %s\n",err_msg);
	}else
		printf("th_int joined: %ld\n",(long int)retval);

	if((status = pthread_join(th_char, (void **)&retval))) {
		strerror_r(status,err_msg,LEN);
		printf("Erro em pthread_join th_char: %s\n",err_msg);
	} else
		printf("th_char joined: %ld\n",(long int)retval);

	if((status = pthread_join(th_struct, (void **)&retval))) {
		strerror_r(status,err_msg,LEN);
		printf("Erro em pthread_join th_struct: %s\n",err_msg);
	} else
		printf("th_struct joined: %ld\n",(long int)retval);

	pthread_exit(NULL);
}

Writing t4.c


In [ ]:
! if [ ! t4 -nt t4.c ]; then gcc -Wall t4.c -o t4 -pthread; fi
! ./t4

Aloˆ, aqui é 1
Aqui é thread dois
Ola' da thread 3 quatro
th_int joined: 1
th_char joined: 2
th_struct joined: 34


#Faça você mesmo
Preencha o código abaixo para torná-lo paralelo:

In [39]:
%%writefile t5.c

/*
  Objetivo: realizar a soma dos elementos de um vetor paralelamente.
  Preencha as lacunas do código.
*/

#include <stdio.h>
#include <stdlib.h>
#include <pthread.h>
#include <sys/time.h>

#define N 100000000

double *vetor;
int nthreads;

typedef struct par {
    int start;
    int end;
    double parcial;
} st_par;

void* soma_parcial(void* arg) {
   st_par *data = ---------- arg;
    data->parcial = 0.0;

    for (int i = data->start; i < data->end; i++) {
        data->parcial += vetor[i];
    }

    pthread_exit(--------);
}

/*função para medição do tempo de execução, não de tempo de cpu como o clock fazia*/
double tempo() {
    struct timeval t;
    gettimeofday(&t, NULL);
    return t.tv_sec + t.tv_usec / 1e6;
}

int main(int argc, char *argv[]) {
    if (argc < 2) {
        printf("Uso: %s <num_threads>\n", argv[0]);
        return 1;
    }

    nthreads = atoi(argv[1]);

    vetor = malloc(sizeof(double) * N);

    for (int i = 0; i < N; i++)
        vetor[i] = 1.0;


 /***** código sequencial simples *****/

    double inicio = tempo();
    double soma_seq = 0.0;

    for (int i = 0; i < N; i++)
        soma_seq += vetor[i];

    double fim = tempo();
    double tempo_seq = fim - inicio;

    printf("Sequencial: %f (tempo = %f s)\n", soma_seq, tempo_seq);



  /***** código paralelo: preencha as lacunas.*****/

    pthread_t threads[---------];
    st_par dados[---------];

    /*um bloco de trabalho ou de dados atribuído a uma thread/processo
    --> termo usado em PPD*/
    int chunk = N / ---------;

    inicio = tempo();
    for (int t = 0; t < ----- ; t++) {

        dados[t].start = t * chunk;

        if (t == nthreads - 1) {
           dados[t].end = N;
        } else {
            dados[t].end = (t + 1) * chunk;
        }

        pthread_create(------, NULL, soma_parcial, --------);
    }

    for (int t = 0; t < nthreads; t++)
        pthread_join(threads[t], NULL);

    double soma_par = 0.0;
    for (int t = 0; t < nthreads; t++)
        soma_par += dados[t].parcial;

    fim = tempo();
    double tempo_par = fim - inicio;

    printf("Paralelo: %f (tempo = %f s)\n", soma_par, tempo_par);
    printf("Speedup: %f\n", tempo_seq / tempo_par);

    free(vetor);
    return 0;
}

Overwriting t5.c


In [40]:
! if [ ! t5 -nt t5.c ]; then gcc -Wall t5.c -o t5 -pthread; fi
! ./t5 2

Sequencial: 100000000.000000 (tempo = 0.359199 s)
Paralelo: 100000000.000000 (tempo = 0.257614 s)
Speedup: 1.394331
